In [ ]:
import os

!git clone https://github.com/fagnerrs/UniMed-CLIP.git
%cd UniMed-CLIP
!pip install -r requirements.txt -q

Cloning into 'UniMed-CLIP'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (167/167), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 167 (delta 58), reused 137 (delta 44), pack-reused 0 (from 0)
Receiving objects: 100% (167/167), 8.32 MiB | 32.13 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/content/UniMed-CLIP
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00


In [ ]:
import sys
sys.path.insert(0, '/content/UniMed-CLIP/src')  # Colab root is always /content

import open_clip
print(open_clip.__file__)

from open_clip import create_model_and_transforms, get_mean_std, HFTokenizer
from unittest.mock import patch

print(open_clip.list_models())

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


/content/UniMed-CLIP/src/open_clip/__init__.py
['RN50', 'RN50-quickgelu', 'RN50x4', 'RN50x16', 'RN101', 'RN101-quickgelu', 'timm-efficientnetv2_rw_s', 'timm-resnet50d', 'timm-resnetaa50d', 'timm-resnetblur50', 'timm-swin_base_patch4_window7_224', 'timm-vit_base_patch16_224', 'timm-vit_base_patch32_224', 'timm-vit_small_patch16_224', 'ViT-B-16', 'ViT-B-16-plus', 'ViT-B-16-plus-240', 'ViT-B-16-quickgelu', 'ViT-B-32', 'ViT-B-32-plus-256', 'ViT-B-32-quickgelu', 'ViT-bigG-14-quickgelu', 'ViT-g-14', 'ViT-H-14', 'ViT-H-14-quickgelu', 'ViT-H-16', 'ViT-L-14', 'ViT-L-14-280', 'ViT-L-14-336', 'ViT-L-14-336-quickgelu', 'ViT-L-14-quickgelu', 'ViT-L-16', 'ViT-L-16-320']


In [ ]:
import torch
from huggingface_hub import hf_hub_download


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")  # confirm GPU is available in Colab runtime

ckpt_path = hf_hub_download(
    repo_id='UzairK/unimed-clip-vit-b16',
    filename='unimed-clip-vit-b16.pt'
)

MODEL_ARCH        = 'ViT-B-16-quickgelu'
text_encoder_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"
mean, std         = get_mean_std()


# Temporarily monkey-patch torch.load to use weights_only=False
original_load = torch.load

def patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return original_load(*args, **kwargs)

with patch('torch.load', patched_load):
    model, _, preprocess = create_model_and_transforms(
        MODEL_ARCH,
        pretrained=ckpt_path,
        precision='amp',
        device=device,
        force_quick_gelu=True,
        mean=mean,
        std=std,
        inmem=True,
        text_encoder_name=text_encoder_name
    )

tokenizer = HFTokenizer(text_encoder_name)
model     = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'UniMed-CLIP ({MODEL_ARCH}) loaded — {n_params/1e6:.1f}M parameters')

Device: cuda


unimed-clip-vit-b16.pt: reconstructing file:   0%|          |  0.00B / 2.35GB            

unimed-clip-vit-b16.pt: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/225k [00:00<?, ?B/s]

UniMed-CLIP (ViT-B-16-quickgelu) loaded — 195.9M parameters


### Fine tunning

In [ ]:
def freeze_layers(model, n_image, n_text) -> None:
  # Start by freezing everything
  for param in model.parameters():
      param.requires_grad = False


  # ── Unfreeze last N image encoder blocks + projection heads ────────
  _unfreeze_last_n_blocks(model, n_image)
  _unfreeze_last_n_text_blocks(model, n_text)
  _unfreeze_projections(model)

  _log_param_counts(model)

def _unfreeze_last_n_blocks(model, n: int) -> None:
    """Unfreeze the last N transformer blocks of the image encoder."""
    # Access the transformer blocks of the visual encoder
    # Path: model.visual.transformer.resblocks (list of N blocks)
    try:
        resblocks = model.visual.transformer.resblocks
    except AttributeError:
        print(
            "Could not find model.visual.transformer.resblocks. "
            "Check model architecture with: "
            "[name for name, _ in model.named_parameters()]"
        )
        return

    total_blocks = len(resblocks)
    n = min(n, total_blocks)   # can't unfreeze more blocks than exist

    print(
        f"Image encoder has {total_blocks} blocks. "
        f"Unfreezing last {n} (blocks {total_blocks - n} to {total_blocks - 1})"
    )

    for block in resblocks[-n:]:
        for param in block.parameters():
            param.requires_grad = True

    # Also unfreeze the final LayerNorm after the transformer
    if hasattr(model.visual, "ln_post"):
        for param in model.visual.ln_post.parameters():
            param.requires_grad = True


def _unfreeze_projections(model) -> None:
    """Unfreeze projection heads and logit_scale."""
    # Image projection head
    if hasattr(model.visual, 'proj') and model.visual.proj is not None:
        model.visual.proj.requires_grad = True

    # Text projection is inside text_encoder.proj (Sequential)
    # already handled in _unfreeze_last_n_text_blocks
    # but safe to call again — requires_grad=True is idempotent
    for param in model.text_encoder.proj.parameters():
        param.requires_grad = True

    # Temperature scalar
    if hasattr(model, 'logit_scale'):
        model.logit_scale.requires_grad = True

def _unfreeze_last_n_text_blocks(model, n: int = 2) -> None:
    """
    Unfreeze the last N layers of BiomedBERT text encoder in UniMed-CLIP.

    Confirmed structure:
    model.text_encoder.transformer.encoder.layer  → 12 BertLayers
    model.text_encoder.pooler                     → ClsLastHiddenStatePooler
    model.text_encoder.proj                       → Sequential
    """
    bert_layers = model.text_encoder.transformer.encoder.layer
    total = len(bert_layers)
    n = min(n, total)

    # Unfreeze last N BERT layers
    for layer in bert_layers[-n:]:
        for param in layer.parameters():
            param.requires_grad = True

    # ← REMOVED: model.text_encoder.transformer.pooler is None in this checkpoint
    # UniMed-CLIP uses its own pooler instead (ClsLastHiddenStatePooler)

    # Unfreeze UniMed-CLIP's custom pooler
    for param in model.text_encoder.pooler.parameters():
        param.requires_grad = True

    # Unfreeze projection head → maps BERT output to shared embedding space
    for param in model.text_encoder.proj.parameters():
        param.requires_grad = True

    print(f"✅ BiomedBERT: unfroze last {n}/{total} layers + pooler + proj")

def _log_param_counts(model) -> None:
    """Print trainable vs total parameter counts."""
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = total - trainable
    pct       = 100.0 * trainable / total

    print(
        f"Parameters → "
        f"trainable: {trainable/1e6:.2f}M | "
        f"frozen: {frozen/1e6:.2f}M | "
        f"total: {total/1e6:.2f}M | "
        f"({pct:.1f}% trainable)"
    )

In [ ]:
from torch.utils.data import Dataset
import json
from PIL import Image

class EFASTDataset(Dataset):
    """
    Loads image-caption pairs from a flat JSON:
    {
        "image_filename.jpg": "caption text...",
        ...
    }
    """

    def __init__(self, captions_json: str, images_dir: str,
                 preprocess, tokenizer, context_length: int = 256):

        with open(captions_json, encoding="utf-8") as f:
            data = json.load(f)

        # data is flat: {filename: caption_string}
        self.samples        = list(data.items())   # [(filename, caption), ...]
        self.images_dir     = images_dir
        self.preprocess     = preprocess
        self.tokenizer      = tokenizer
        self.context_length = context_length

        print(f"Dataset loaded: {len(self.samples)} image-caption pairs")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        filename, caption = self.samples[idx]

        img_path = os.path.join(self.images_dir, filename)
        try:
            image = self.preprocess(Image.open(img_path).convert("RGB"))
        except Exception as e:
            raise RuntimeError(f"Could not load image {img_path}: {e}")

        text = self.tokenizer([caption], context_length=self.context_length)[0]

        return image, text

In [ ]:
full_dataset = EFASTDataset(
      captions_json  = "/content/drive/MyDrive/efast-train-dataset/image-descriptions.json",
      images_dir     = "/content/drive/MyDrive/efast-train-dataset",
      preprocess     = preprocess,
      tokenizer      = tokenizer,
  )

Dataset loaded: 20399 image-caption pairs


In [ ]:
n_image_blocks = 10
n_text_blocks = 10

freeze_layers(model, n_image=n_image_blocks, n_text=n_text_blocks)

for name, param in model.named_parameters():
   status = "🟢 TRAIN" if param.requires_grad else "🔴 FROZEN"
   shape  = str(list(param.shape))
   #print(f"{status} | {name:<65} | {shape}")

Image encoder has 12 blocks. Unfreezing last 10 (blocks 2 to 11)
✅ BiomedBERT: unfroze last 10/12 layers + pooler + proj
Parameters → trainable: 142.97M | frozen: 52.93M | total: 195.90M | (73.0% trainable)


In [ ]:
from torch.utils.data import random_split

seed = 42
train_percent = 0.85
validation_percent = 0.15

total = len(full_dataset)
train_size = int(train_percent * total)
val_size = int(validation_percent   * total)
test_size = total - train_size - val_size

train_set, val_set, test_set = random_split(
      full_dataset,
      [train_size, val_size, test_size],
      generator=torch.Generator().manual_seed(seed),
  )

print(
        f"Split → train: {len(train_set)} | "
        f"val: {len(val_set)} | "
        f"test: {len(test_set)}"
    )

Split → train: 17339 | val: 3059 | test: 1


In [ ]:
from torch.utils.data import DataLoader

batch_size = 128
num_workers = 4

train_loader = DataLoader(
    train_set, batch_size=batch_size,
    shuffle=True,  num_workers=num_workers, pin_memory=True
)
val_loader = DataLoader(
    val_set, batch_size=batch_size,
    shuffle=False, num_workers=num_workers, pin_memory=True
)
test_loader = DataLoader(
    test_set, batch_size=batch_size,
    shuffle=False, num_workers=num_workers, pin_memory=True
)

In [ ]:
from torch.cuda.amp import GradScaler

lr = 5e-6
decay = 0.01
epochs = 20

# ── Optimizer ─────────────────────────────────────────────────────────────
# filter() ensures only unfrozen parameters are passed to optimizer
# This is the key line that makes the freeze strategy effective
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=lr,
    weight_decay=decay,
)

# Cosine annealing: smoothly decays lr to ~0 over training
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs,
    eta_min=1e-8,
)

# Mixed precision scaler
scaler = GradScaler()

/tmp/ipykernel_2303/4271088073.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [ ]:
from torch.cuda.amp import autocast
import torch.nn.functional as F

def infonce_loss(image_features: torch.Tensor,
                 text_features: torch.Tensor,
                 logit_scale: torch.Tensor) -> torch.Tensor:
    """
    Symmetric InfoNCE contrastive loss (same as CLIP training objective).

    For a batch of B image-text pairs:
    - Diagonal of the similarity matrix = correct pairs
    - Off-diagonal = negatives (wrong pairings within the batch)

    The loss pushes correct pairs to have high similarity
    and incorrect pairs to have low similarity.

    Args:
        image_features : [B, D] — NOT yet normalized
        text_features  : [B, D] — NOT yet normalized
        logit_scale    : scalar — learned temperature (log scale)

    Returns:
        scalar loss value
    """
    # L2 normalize both feature vectors
    image_features = F.normalize(image_features, dim=-1)
    text_features  = F.normalize(text_features,  dim=-1)

    # Scale factor — clamp to avoid numerical instability
    scale = logit_scale.exp().clamp(max=100)

    # Similarity matrix: [B, B]
    # logits_per_image[i, j] = similarity(image_i, text_j)
    logits_per_image = scale * image_features @ text_features.T   # [B, B]
    logits_per_text  = logits_per_image.T                         # [B, B]

    # Ground truth labels: index i should match index i
    # i.e. image_0 matches text_0, image_1 matches text_1, etc.
    batch_size = image_features.shape[0]
    labels = torch.arange(batch_size, device=image_features.device)

    # Cross-entropy from both directions, averaged
    loss_image = F.cross_entropy(logits_per_image, labels)
    loss_text  = F.cross_entropy(logits_per_text,  labels)
    loss       = (loss_image + loss_text) / 2.0

    return loss

def run_epoch(model, loader, optimizer, scaler, device, is_train: bool):
    """
    Run one full epoch of either training or validation.

    During training:  model weights are updated via backpropagation
    During validation: no weight updates, only loss is computed

    Returns average loss over the epoch.
    """
    model.train() if is_train else model.eval()
    total_loss = 0.0

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for batch_idx, (images, texts) in enumerate(loader):
            images = images.to(device, non_blocking=True)
            texts  = texts.to(device,  non_blocking=True)

            if is_train:
                optimizer.zero_grad()

            # Mixed precision forward pass
            with autocast():
                image_features = model.encode_image(images)
                text_features  = model.encode_text(texts)
                loss = infonce_loss(
                    image_features,
                    text_features,
                    model.logit_scale
                )

            if is_train:
                # Scale loss for mixed precision, backpropagate
                scaler.scale(loss).backward()

                # Unscale before clipping so clip threshold is in real scale
                scaler.unscale_(optimizer)

                # Gradient clipping — prevents exploding gradients
                # Only clips trainable parameters (frozen ones have no grad)
                torch.nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, model.parameters()),
                    max_norm=1.0
                )

                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item()

            # Progress logging every 10 batches
            if batch_idx % 10 == 0:
                phase = "TRAIN" if is_train else "VAL"
                print(
                    f"  [{phase}] batch {batch_idx+1}/{len(loader)} "
                    f"loss={loss.item():.4f}"
                )

    return total_loss / len(loader)


def save_checkpoint(model, optimizer, epoch, val_loss, output_dir, tag="best"):
    """Save model checkpoint with metadata."""
    path = os.path.join(output_dir, f"efast_clip_{tag}.pt")
    torch.save({
        "epoch":                epoch,
        "state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss":             val_loss,
    }, path)
    print(f"Checkpoint saved → {path}")
    return path


In [ ]:
output_dir = "/content/drive/MyDrive/efast-output-dir"

# Create the output directory if it doesn't exist
import os
os.makedirs(output_dir, exist_ok=True)

# ── Training loop ─────────────────────────────────────────────────────────
print(f"Starting training for {epochs} epochs...")
best_val_loss = float("inf")
history = {"train_loss": [], "val_loss": []}

for epoch in range(1, epochs + 1):
    print(f"\n{'─'*60}")
    print(f"Epoch {epoch}/{epochs}  |  lr={scheduler.get_last_lr()}")
    print(f"{'─'*60}")

    # Training pass
    train_loss = run_epoch(
        model, train_loader, optimizer, scaler, device, is_train=True
    )

    # Validation pass
    val_loss = run_epoch(
        model, val_loader, optimizer, scaler, device, is_train=False
    )

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    print(
        f"Epoch {epoch:02d} summary → "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f}"
    )

    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model, optimizer, epoch, val_loss,
                        output_dir, tag="best")
        print(f"✅ New best model (val_loss={val_loss:.4f})")

    # Save latest checkpoint every epoch (useful for resuming)
    save_checkpoint(model, optimizer, epoch, val_loss,
                    output_dir, tag="latest")

# ── Save training history ─────────────────────────────────────────────────
history_path = os.path.join(output_dir, "training_history.json")

with open(history_path, "w") as f:
    json.dump(history, f, indent=2)

print(f"Training history saved → {history_path}")

Starting training for 20 epochs...

────────────────────────────────────────────────────────────
Epoch 1/20  |  lr=[5e-06]
────────────────────────────────────────────────────────────


/tmp/ipykernel_2303/4177156352.py:72: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  [TRAIN] batch 1/136 loss=6.4438
  [TRAIN] batch 11/136 loss=4.9223
  [TRAIN] batch 21/136 loss=4.3203
  [TRAIN] batch 31/136 loss=4.1043
  [TRAIN] batch 41/136 loss=3.7298
  [TRAIN] batch 51/136 loss=3.6882
  [TRAIN] batch 61/136 loss=3.6618
  [TRAIN] batch 71/136 loss=3.5147
  [TRAIN] batch 81/136 loss=3.4733
  [TRAIN] batch 91/136 loss=3.4247
  [TRAIN] batch 101/136 loss=3.3309
  [TRAIN] batch 111/136 loss=3.2644
  [TRAIN] batch 121/136 loss=3.2825
  [TRAIN] batch 131/136 loss=3.2763
  [VAL] batch 1/24 loss=3.0644
  [VAL] batch 11/24 loss=3.0736
  [VAL] batch 21/24 loss=3.1446
Epoch 01 summary → train_loss=3.8262 | val_loss=3.1199
Checkpoint saved → /content/drive/MyDrive/efast-output-dir/efast_clip_best.pt
✅ New best model (val_loss=3.1199)
Checkpoint saved → /content/drive/MyDrive/efast-output-dir/efast_clip_latest.pt

────────────────────────────────────────────────────────────
Epoch 2/20  |  lr=[4.969282409784869e-06]
────────────────────────────────────────────────────────────